# SANA A-PSL: Few-Shot Multilingual Transfer Learning on Pakistani Sign Language (PSL)
**Goal:** Fine-tune our pre-trained Conv1D Foundation Model (`best_how2sign_phase2_model.pt`) on the verified 71-word Pakistani Sign Language dataset.
- **Data Input:** MediaPipe 3D Hand Landmarks from Laptop, Mobile, and Webcams (`126-dim` -> `208-dim` SANA adapter).
- **Output:** Bilingual translations in **English** and **Urdu (`اردو`)** with sub-300ms latency.

In [ ]:
# ── Cell 1: Environment Setup, Dependencies & Config ─────────────────────────
!pip install -q transformers peft wandb
import os, sys, glob, random, math, time, gc, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
from transformers.modeling_outputs import BaseModelOutput
import wandb

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# ── Locate Pre-Trained Foundation Checkpoint ──
base_checkpoint = None
for root, dirs, files in os.walk("/kaggle"):
    if "best_how2sign_phase2_model.pt" in files:
        base_checkpoint = os.path.join(root, "best_how2sign_phase2_model.pt")
        break

CONFIG = {
    "BASE_DIR":         "/kaggle/input/datasets/mohib123456/dynamic-word-level-pakistan-sign-language-dataset/PakistanSignLanguageDatasetV2/PakistanSignLanguageDatasetV2/laptop_data",
    "BASE_CHECKPOINT":  base_checkpoint,
    "MT5_MODEL_NAME":   "google/mt5-small",
    "D_MODEL":          512,
    "NUM_HEADS":        8,
    "NUM_ENCODER_LAYERS": 2,
    "DIM_FEEDFORWARD":  1024,
    "DROPOUT":          0.108,
    "MAX_SEQ_LEN":      100,
    "MAX_TARGET_LEN":   32,
    "INPUT_DIM":        208,
    "TARGET_LANGUAGE":  "urdu",      # 'urdu' or 'english'
    "EPOCHS":           5,
    "BATCH_SIZE":       16,
    "LEARNING_RATE":    1e-4,
    "WEIGHT_DECAY":     1e-4,
}

print("Foundation Checkpoint Found:", base_checkpoint)
print("PSL Training Configuration Initialized.")

In [ ]:
# ── Cell 2: Bilingual PSL Dictionary & Dataset Loader ────────────────────────
# Mapping PSL Classes to English Sentences & Urdu Script
BILINGUAL_DICT = {
    "assalam-o-alaikum": ("Assalam-o-Alaikum", "السلام علیکم"),
    "absolutely":        ("Absolutely", "بالکل"),
    "aircrash":          ("Air crash", "ہوائی حادثہ"),
    "airplane":          ("Airplane", "ہوائی جہاز"),
    "all":               ("All of them", "تمام"),
    "also":              ("Also", "بھی"),
    "arrival":           ("Arrival", "آمد"),
    "atm":               ("ATM machine", "اے ٹی ایم"),
    "bald":              ("Bald head", "گنجا"),
    "beach":             ("Beach", "ساحل سمندر"),
    "beak":              ("Beak", "چونچ"),
    "bear":              ("Bear", "ریچھ"),
    "beard":             ("Beard", "داڑھی"),
    "bed":               ("Bed", "بستر"),
    "bench":             ("Bench", "بینچ"),
    "hungry":            ("I am hungry", "مجھے بھوک لگی ہے"),
    "excuseme":          ("Excuse me", "معاف کیجیے گا"),
    "we":                ("We are here", "ہم سب"),
    "mine":              ("This is mine", "یہ میرا ہے"),
    "fan":               ("Fan", "پنکھا"),
    "facelotion":        ("Face lotion", "فیس لوشن"),
    "left_hand":         ("Left hand", "بایاں ہاتھ"),
}

print("Indexing all 71 PSL sequences...")
all_samples = []
word_classes = sorted([d for d in os.listdir(CONFIG["BASE_DIR"]) if os.path.isdir(os.path.join(CONFIG["BASE_DIR"], d))])

for word in word_classes:
    word_dir = os.path.join(CONFIG["BASE_DIR"], word)
    seq_dirs = [os.path.join(word_dir, s) for s in os.listdir(word_dir) if os.path.isdir(os.path.join(word_dir, s))]
    
    en_text, urdu_text = BILINGUAL_DICT.get(word.lower(), (word.replace('_', ' ').title(), word))
    
    for s_dir in seq_dirs:
        frame_files = sorted(glob.glob(os.path.join(s_dir, "*.npy")))
        if len(frame_files) >= 10:
            all_samples.append((frame_files, en_text, urdu_text))

print(f"Successfully indexed {len(all_samples)} total sign sequences across {len(word_classes)} PSL classes!")

class PSLSignDataset(Dataset):
    def __init__(self, samples_list, tokenizer, config):
        self.samples = samples_list
        self.tokenizer = tokenizer
        self.config = config

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame_files, en_text, urdu_text = self.samples[idx]
        target_text = urdu_text if self.config["TARGET_LANGUAGE"] == "urdu" else en_text
        
        # Load frames (T, 126)
        raw_frames = [np.load(f) for f in frame_files]
        raw = np.array(raw_frames, dtype=np.float32)
        T = raw.shape[0]
        
        # Extract 2D Hand Landmarks: Left Hand (63 -> 42) & Right Hand (63 -> 42)
        lh_3d = raw[:, :63].reshape(T, 21, 3)
        rh_3d = raw[:, 63:126].reshape(T, 21, 3)
        lh_2d = lh_3d[:, :, :2].reshape(T, 42)
        rh_2d = rh_3d[:, :, :2].reshape(T, 42)
        
        # Attach 66 Pose (Neutral) + 42 LH + 42 RH + 58 Face (Neutral) = 208 floats
        pose_66 = np.zeros((T, 66), dtype=np.float32)
        face_58 = np.zeros((T, 58), dtype=np.float32)
        adapted_208 = np.concatenate([pose_66, lh_2d, rh_2d, face_58], axis=1)
        adapted_208 = np.nan_to_num(adapted_208, nan=0.0, posinf=0.0, neginf=0.0)
        
        # Pad / Truncate to MAX_SEQ_LEN
        max_len = self.config["MAX_SEQ_LEN"]
        if T >= max_len:
            padded_seq = adapted_208[:max_len]
            valid_len = max_len
        else:
            padded_seq = np.zeros((max_len, 208), dtype=np.float32)
            padded_seq[:T] = adapted_208
            valid_len = T
            
        attention_mask = np.zeros(max_len, dtype=np.float32)
        attention_mask[:valid_len] = 1.0
        
        tokens = self.tokenizer(
            target_text, max_length=self.config["MAX_TARGET_LEN"], padding="max_length", truncation=True, return_tensors="pt"
        )
        labels = tokens["input_ids"].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            "input_ids": torch.tensor(padded_seq, dtype=torch.float32),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.float32),
            "labels": labels,
            "raw_text": target_text
        }

tokenizer = AutoTokenizer.from_pretrained(CONFIG["MT5_MODEL_NAME"])

# 80% Train, 20% Validation Split
train_size = int(0.8 * len(all_samples))
val_size   = len(all_samples) - train_size
train_samples, val_samples = random_split(all_samples, [train_size, val_size])

train_dataset = PSLSignDataset(train_samples, tokenizer, CONFIG)
val_dataset   = PSLSignDataset(val_samples, tokenizer, CONFIG)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=2)

print(f"DataLoaders Ready: {len(train_dataset)} Train sequences | {len(val_dataset)} Validation sequences.")

In [ ]:
# ── Cell 3: Conv1D SpatialTemporal Architecture & Weight Injection ────────────
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TemporalGestureTokenizer(nn.Module):
    def __init__(self, input_dim=208, d_model=512):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, d_model // 2, kernel_size=5, stride=2, padding=2)
        self.norm1 = nn.BatchNorm1d(d_model // 2)
        self.gelu  = nn.GELU()
        self.conv2 = nn.Conv1d(d_model // 2, d_model, kernel_size=5, stride=2, padding=2)
        self.norm2 = nn.BatchNorm1d(d_model)
        
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.gelu(self.norm1(self.conv1(x)))
        x = self.gelu(self.norm2(self.conv2(x)))
        return x.transpose(1, 2)

class UpgradedSpatialTemporalEncoder(nn.Module):
    def __init__(self, input_dim, d_model, num_heads, num_layers, ffn_dim, dropout, max_len):
        super().__init__()
        self.tokenizer = TemporalGestureTokenizer(input_dim, d_model)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model, max_len=100)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=ffn_dim,
            dropout=dropout, activation="gelu", batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src, src_key_padding_mask=None):
        x = self.tokenizer(src)
        x = self.pos_encoder(x)
        if src_key_padding_mask is not None:
            downsampled_mask = src_key_padding_mask[:, ::4]
            if downsampled_mask.size(1) != x.size(1):
                downsampled_mask = downsampled_mask[:, :x.size(1)]
        else:
            downsampled_mask = None
        out = self.transformer_encoder(x, src_key_padding_mask=downsampled_mask)
        return self.norm(out)

class SANA_PSL_Translator(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.visual_encoder = UpgradedSpatialTemporalEncoder(
            input_dim=config["INPUT_DIM"],
            d_model=config["D_MODEL"],
            num_heads=config["NUM_HEADS"],
            num_layers=config["NUM_ENCODER_LAYERS"],
            ffn_dim=config["DIM_FEEDFORWARD"],
            dropout=config["DROPOUT"],
            max_len=config["MAX_SEQ_LEN"]
        )
        print(f"Loading multilingual language backbone: {config['MT5_MODEL_NAME']} (Urdu + English)...")
        self.mt5 = AutoModelForSeq2SeqLM.from_pretrained(config["MT5_MODEL_NAME"])
        
        # Freeze base mT5 except cross-attention layers
        for param in self.mt5.parameters():
            param.requires_grad = False
            
        for block in self.mt5.decoder.block:
            for param in block.layer[1].parameters():
                param.requires_grad = True
                
    def forward(self, input_ids, attention_mask, labels=None):
        key_padding_mask = (attention_mask == 0)
        encoder_hidden_states = self.visual_encoder(input_ids, src_key_padding_mask=key_padding_mask)
        downsampled_mask = attention_mask[:, ::4]
        if downsampled_mask.size(1) != encoder_hidden_states.size(1):
            downsampled_mask = downsampled_mask[:, :encoder_hidden_states.size(1)]
        encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)
        return self.mt5(encoder_outputs=encoder_outputs, attention_mask=downsampled_mask, labels=labels)
        
    def generate(self, input_ids, attention_mask, max_length=32):
        key_padding_mask = (attention_mask == 0)
        encoder_hidden_states = self.visual_encoder(input_ids, src_key_padding_mask=key_padding_mask)
        downsampled_mask = attention_mask[:, ::4]
        if downsampled_mask.size(1) != encoder_hidden_states.size(1):
            downsampled_mask = downsampled_mask[:, :encoder_hidden_states.size(1)]
        encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)
        return self.mt5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=downsampled_mask,
            max_new_tokens=20,
            do_sample=True,
            temperature=0.7,
            top_p=0.88,
            repetition_penalty=1.6,
            no_repeat_ngram_size=2
        )

model = SANA_PSL_Translator(CONFIG).to(device)

# Warm-Start Pre-trained Weights from Foundation Model
if CONFIG["BASE_CHECKPOINT"] and os.path.exists(CONFIG["BASE_CHECKPOINT"]):
    print(f"Injecting pre-trained Conv1D Foundation weights from: {CONFIG['BASE_CHECKPOINT']}...")
    ckpt = torch.load(CONFIG["BASE_CHECKPOINT"], map_location=device)
    state_dict = ckpt.get("model_state_dict", ckpt)
    model.load_state_dict(state_dict, strict=False)
    print("Pre-trained weights injected successfully!")
else:
    print("Note: Training from fresh base initialization.")

In [ ]:
# ── Cell 4: Fast Few-Shot Training Loop ──────────────────────────────────────
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["LEARNING_RATE"],
    weight_decay=CONFIG["WEIGHT_DECAY"]
)
total_steps = len(train_loader) * CONFIG["EPOCHS"]
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

print(f"Beginning Few-Shot PSL Fine-Tuning ({CONFIG['EPOCHS']} Epochs, {total_steps} Total Steps)...")
best_val_loss = float("inf")

for epoch in range(1, CONFIG["EPOCHS"] + 1):
    model.train()
    running_loss = 0.0
    t0 = time.time()
    
    for step, batch in enumerate(train_loader, 1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
        
    train_loss = running_loss / len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            
    val_loss /= len(val_loader)
    elapsed = time.time() - t0
    
    print(f"Epoch [{epoch}/{CONFIG['EPOCHS']}] ({elapsed:.1f}s) | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_path = "/kaggle/working/best_sana_psl_model.pt"
        torch.save({"model_state_dict": model.state_dict(), "val_loss": val_loss}, save_path)
        print(f"  --> [SAVED BEST PSL MODEL] -> {save_path}")

In [ ]:
# ── Cell 5: Live Bilingual Translation Test (English & Urdu) ─────────────────
print("=" * 80)
print("SANA A-PSL LIVE TRANSLATION VERIFICATION (Pakistani Sign Language)")
print("=" * 80)

model.eval()
val_batch = next(iter(val_loader))
input_ids = val_batch["input_ids"].to(device)
attention_mask = val_batch["attention_mask"].to(device)
ground_truths = val_batch["raw_text"]

with torch.no_grad():
    for i in range(min(5, len(ground_truths))):
        t0 = time.time()
        gen_ids = model.generate(input_ids[i:i+1], attention_mask[i:i+1])
        latency_ms = (time.time() - t0) * 1000
        
        pred_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
        print(f"\nSample #{i+1}:")
        print(f"  [Ground Truth]: \"{ground_truths[i]}\"")
        print(f"  [SANA AI]:      \"{pred_text}\"")
        print(f"  [Latency]:      {latency_ms:.1f} ms")
        print("-" * 60)